# 03 · Training

Training repeats one small step thousands of times:

1. Show the model a **batch** of 64 scenes.
2. Measure how wrong its outputs are with a single number, the **loss**.
3. Work out, for each of the 588,556 weights, which direction would lower the loss (`loss.backward()`).
4. Nudge every weight a little in that direction (`optimizer.step()`).

A full run is 20 **epochs** (passes) of 20,000 scenes, which is 313 steps per epoch and 6,260 steps in all.
It takes about an hour on an M-series Mac, so this notebook doesn't run one inline. It shows every piece on a
small scale, then gives you a cell that starts the real run in the background.

**You'll see:** what the loss measures, a model memorizing 32 scenes (the standard "is the pipeline wired
right?" check), a 1-minute training run, the learning curves of past runs, and how to start a full run.

**Needs:** raw GTSRB and COCO downloads (fresh training scenes are built from them), `data/processed/val.npz`.
**Writes:** nothing, unless `FULL_RUN` is on (then a new run in `outputs/checkpoints/` and `outputs/runs/`).

In [ ]:
# ---- Settings: the only cell you need to edit ----
CONFIG = "configs/peak_fresh_aug.yaml"  # the recipe to train; configs/ has one file per project version
FULL_RUN = False  # True: start a full ~1-hour training run in the background (last section)
QUICK_SCENES = 3000  # size of the mini training run in section 3 (fresh scenes per epoch)
QUICK_EPOCHS = 3

In [ ]:
# Setup (the same in every notebook). All paths in this project (configs/, data/, outputs/) are relative
# to the repo root, and the scripts expect to run from there, so move this notebook's working folder up
# to the root first. Safe to run twice.
import os
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists():  # walk up until we reach the folder holding pyproject.toml
    if root == root.parent:
        raise RuntimeError("can't find the repo root: open this notebook from inside FindTheObject-CNN-Project")
    root = root.parent
os.chdir(root)
if str(root / "notebooks") not in sys.path:
    sys.path.insert(0, str(root / "notebooks"))  # lets us import notebooks/nbhelpers.py

from nbhelpers import NB_OUT, ROOT, find_run, read_json, require, run_script
print("working in", ROOT)

In [ ]:
# Stop here, with a clear message, if a file this notebook reads is missing.
require("data/raw/gtsrb/GTSRB/Final_Training/Images", "data/raw/coco/val2017",
        how="download them as described in README.md (Data)")
require("data/processed/val.npz", how="python scripts/make_dataset.py --config configs/baseline.yaml")

In [ ]:
# Read the config: the train: section becomes a TrainConfig, which rejects missing, unknown or bad values.
import yaml

from findobject.models.factory import Arch
from findobject.training.config import TrainConfig
from findobject.utils.device import pick_device

raw_cfg = yaml.safe_load((ROOT / CONFIG).read_text())
cfg = TrainConfig(**raw_cfg["train"])
arch = Arch(raw_cfg["model"]["arch"])
device = pick_device(cfg.device)
print(cfg)
print("arch:", arch.value, "| device:", device)

## 1. The loss: one number for "how wrong"

`detection_loss` adds two parts:

* **Class loss** (cross-entropy): `-log(probability the model gave the true class)`. It's 0 for a confident
  right answer and large for a confident wrong one. A model that guesses evenly among 43 classes scores
  `ln(43) = 3.76`.
* **Heatmap loss** (binary cross-entropy, added up over all 1,024 cells): for every cell, how far the predicted
  probability is from the target heatmap. The target bump has soft edges (values like 0.3), so even a perfect
  prediction can't reach 0. That lowest possible value is the **floor**, and `heatmap_loss - floor` (the
  "excess") is the part the model can still remove.

`total = class_loss + 0.1 × heatmap_loss`. The 0.1 keeps the heatmap part (about 39 at the start) from
drowning out the class part (about 3.8).

In [ ]:
# Score one batch of 64 training scenes with an untrained model and compare each part with what we expect.
import math

import torch
from torch.utils.data import DataLoader, Subset, default_collate

from findobject.models.factory import build_model
from findobject.training.losses import detection_loss
from findobject.training.train_data import epoch_datasets
from findobject.utils.seed import seed_everything

shuffle_gen = seed_everything(cfg.seed)  # seeds Python, numpy and torch; returns the generator used for shuffling
for_epoch = epoch_datasets(raw_cfg["data"], cfg.seed)
batch = default_collate([for_epoch(1)[i] for i in range(64)]).to(device)  # 64 scenes stacked into one batch

model = build_model(arch).to(device)
with torch.no_grad():
    parts = detection_loss(model(batch.image), batch, cfg.heatmap_loss_weight)
print(f"class loss   {parts.class_loss:.3f}   (an even guess gives ln 43 = {math.log(43):.3f})")
print(f"heatmap loss {parts.heatmap_loss:.2f}   (floor {parts.heatmap_floor:.2f}, so excess "
      f"{parts.heatmap_loss - parts.heatmap_floor:.2f})")
print(f"total        {parts.total:.3f}   = class + {cfg.heatmap_loss_weight} x heatmap")

## 2. Sanity check: can it memorize 32 scenes?

Before spending an hour on training, check that the pipeline can learn at all. Give the model the **same 32
scenes 300 times**. A correctly wired model memorizes them perfectly. If it can't, something is broken
(labels shuffled against images, a wrong loss, a frozen layer), and no amount of training would help.
This is the same check as `python scripts/train.py --overfit 32`.

In [ ]:
# Train on 32 fixed scenes for 300 steps (~15 s), recording the loss and the accuracy on those same 32 scenes.
from torch.optim import AdamW

from findobject.eval.evaluate import evaluate

tiny = default_collate([for_epoch(1)[i] for i in range(32)]).to(device)
torch.manual_seed(cfg.seed)
model = build_model(arch).to(device)
optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
history = []
for step in range(1, 301):
    model.train()  # train mode: BatchNorm uses this batch's statistics
    parts = detection_loss(model(tiny.image), tiny, cfg.heatmap_loss_weight)
    optimizer.zero_grad(set_to_none=True)  # forget the previous step's gradients
    parts.total.backward()  # compute every weight's gradient
    optimizer.step()  # nudge every weight
    if step == 1 or step % 25 == 0:
        r = evaluate(model, [tiny], device, cfg.heatmap_loss_weight)  # eval mode, same 32 scenes
        history.append((step, parts.class_loss.item(), (parts.heatmap_loss - parts.heatmap_floor).item(),
                        r.class_acc, r.pointing_acc))
        print(f"step {step:3}  class loss {history[-1][1]:.3f}  heatmap excess {history[-1][2]:6.2f}  "
              f"class acc {r.class_acc:.2f}  pointing acc {r.pointing_acc:.2f}")
print("PASS: memorized all 32" if r.class_acc == 1 and r.pointing_acc == 1 else "FAIL: look for a pipeline bug")

In [ ]:
# Plot the two losses falling as the model memorizes (log scale: they fall by orders of magnitude).
import matplotlib.pyplot as plt

steps, cls_loss, excess, *_ = zip(*history)
plt.figure(figsize=(7, 3))
plt.semilogy(steps, cls_loss, label="class loss"), plt.semilogy(steps, excess, label="heatmap excess")
plt.xlabel("step"), plt.legend(), plt.title("memorizing 32 scenes")
plt.show()

## 3. A one-minute training run

Now real training, just smaller: `QUICK_SCENES` fresh scenes per epoch for `QUICK_EPOCHS` epochs, then a score
on the first 1,000 val scenes after each epoch. It uses the same functions `scripts/train.py` uses:
`train_one_epoch` for the steps, `evaluate` for the score.

The **learning rate** (the size of each nudge) starts at 0.001 and falls smoothly to 0 along a cosine curve over
the whole run, with big steps early and fine adjustments at the end. A full run follows the same curve over
6,260 steps.

In [ ]:
# Train a fresh model on QUICK_SCENES new scenes per epoch, scoring it on 1,000 val scenes after each epoch.
import time

from torch.optim.lr_scheduler import CosineAnnealingLR

from findobject.data.dataset import SignScenes
from findobject.training.epoch import train_one_epoch

val_loader = DataLoader(Subset(SignScenes(ROOT / "data/processed/val.npz"), range(1000)), batch_size=256)
torch.manual_seed(cfg.seed)
model = build_model(arch).to(device)
optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
steps_per_epoch = math.ceil(QUICK_SCENES / cfg.batch_size)
scheduler = CosineAnnealingLR(optimizer, T_max=QUICK_EPOCHS * steps_per_epoch)
lrs = []  # the learning rate after every step, to plot below
curve = []
for epoch in range(1, QUICK_EPOCHS + 1):
    t0 = time.time()
    train = DataLoader(Subset(for_epoch(epoch), range(QUICK_SCENES)), batch_size=cfg.batch_size, shuffle=True,
                       generator=shuffle_gen)  # num_workers=0: scenes are made in this process
    stats = train_one_epoch(model, train, optimizer, scheduler, device, cfg.heatmap_loss_weight,
                            on_step=lambda parts: lrs.append(scheduler.get_last_lr()[0]))
    val = evaluate(model, val_loader, device, cfg.heatmap_loss_weight)
    curve.append((epoch, stats.class_acc, val.class_acc, val.pointing_acc, val.joint_acc))
    print(f"epoch {epoch} ({time.time() - t0:.0f}s)  train: loss {stats.mean_total:.3f} class {stats.class_acc:.3f}"
          f"  |  val: class {val.class_acc:.3f} pointing {val.pointing_acc:.3f} joint {val.joint_acc:.3f}")

In [ ]:
# Left: the learning rate over the mini run. Right: val accuracy after each epoch.
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(lrs), axes[0].set_xlabel("step"), axes[0].set_title("learning rate (cosine decay)")
ep, _, v_cls, v_pt, v_joint = zip(*curve)
for ys, label in ((v_cls, "class"), (v_pt, "pointing"), (v_joint, "joint")):
    axes[1].plot(ep, ys, marker="o", label=label)
axes[1].set_xlabel("epoch"), axes[1].set_xticks(ep), axes[1].legend(), axes[1].set_title("val accuracy")
plt.show()

Pointing is learned first: finding "the odd thing in the picture" is easier than telling 43 classes apart. Class
accuracy here stays far below the ~98% of a full run, which sees about 45 times as many scene-steps.

## 4. Learning curves of the real runs

Every full run logs to TensorBoard (`outputs/runs/<run>/`). We can read those logs directly. Here's val joint
accuracy per epoch for one run of each version:

In [ ]:
# Read val/joint_acc from each run's TensorBoard log and plot them together.
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

VERSIONS = {"baseline": "v1", "peak": "v1.5", "peak_fresh_aug": "v1.5b", "peak_empty": "v3a", "peak_decoy": "v3a.1"}
plt.figure(figsize=(8, 4))
for name, version in VERSIONS.items():
    log = EventAccumulator(str(ROOT / "outputs/runs" / find_run(name).name)).Reload()  # same folder name as the checkpoint
    events = log.Scalars("val/joint_acc")
    plt.plot([e.step for e in events], [e.value for e in events], marker=".", label=f"{version} ({name})")
plt.ylim(0.85, 1), plt.xlabel("epoch"), plt.ylabel("val joint accuracy"), plt.legend(), plt.grid(alpha=0.3)
plt.show()

To explore every logged curve, image panel and confusion matrix interactively, open TensorBoard. In VS Code,
open the command palette (Cmd+Shift+P) and choose **Python: Launch TensorBoard**, pointing it at
`outputs/runs`. Or, in a terminal: `tensorboard --logdir outputs/runs`.

## Run it for real: a full training run

Two steps:

1. `--overfit 32`: the memorization check from section 2, run by the real script. It takes about a minute
   and fails loudly if something is broken.
2. The full run, started **in the background**. It keeps going if you close this notebook, restart the kernel
   or quit VS Code (sleeping the Mac pauses it). Its output goes to a log file, and the status cell below shows
   the latest lines.

When it finishes, its checkpoint is in `outputs/checkpoints/<run_name>-<date>-<time>/`. To look at it in
notebooks 04–08, set `RUN_NAME` in their Settings cell to the config's `run_name` (here: `peak_fresh_aug`).
They always pick the newest run with that name. To keep an older run as the default instead, copy the config
and give it a new `run_name` before training.

In [ ]:
# Step 1: the overfit sanity check via the real script (~1 min). Only runs when FULL_RUN = True.
run_script("train", "--config", CONFIG, "--overfit", 32, enabled=FULL_RUN)

In [ ]:
# Step 2: start the full run in the background. Running this cell twice won't start a second copy.
from nbhelpers import background_status, start_background

LOG = NB_OUT / f"train-{cfg.run_name}.log"
if FULL_RUN:
    start_background("train", "--config", CONFIG, log=LOG)
else:
    print(f"$ python scripts/train.py --config {CONFIG}   (in the background, log: {LOG})")
    print("  (not run: set FULL_RUN = True in the Settings cell to run it)")

In [ ]:
# Check on the background run: still going or finished, and the last lines it printed. Re-run any time.
if LOG.with_suffix(".pid").exists():
    background_status(LOG)
else:
    print("no run started from this notebook yet")

**Next:** `04_evaluate_synthetic.ipynb`, scoring a trained model.